In [1]:
import sys
import json
import numpy as np
from numpy import float64
import scipy
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy
from sympy import *
from sympy import zoo, oo, nan
from sympy.core.numbers import ComplexInfinity
from random import seed, random, choice
from itertools import product, permutations
from scipy.optimize import curve_fit, least_squares, minimize
from scipy.integrate import solve_ivp
from numba import njit
from scipy.signal import savgol_filter

In [2]:
def make_sse_function_from_ode(ode_func, t_eval,int, x0_fixed=None, param_fixed=None,
                                   optimize_x0=False, optimize_params=False, observed=None, verbose=False):
    x0_len = len(x0_fixed) if x0_fixed is not None else 0
    param_len = len(param_fixed) if param_fixed is not None else 0
    def sse_fun(x):
        idx = 0
        if optimize_x0:
            x0 = x[idx:idx + x0_len]
            idx += x0_len
        else:
            x0 = x0_fixed
    
        if optimize_params:
            params = x[idx:idx + param_len]
        else:
            params = param_fixed
        x0,params = np.asarray(x0, dtype=float64) ,np.asarray(params, dtype=float64)
        try:
            sol = int(ode_func, t_eval, x0, params)
        except Exception as e:
            if verbose:
                print("Integration error:", e)
            # Return large residuals to discourage invalid regions
            return np.inf
    
        try:
            #residuals = (sol - observed).ravel()  # Flattened vector
            sse = np.sum((observed - sol)**2.)
            #residuals[np.isnan(residuals)] = 1e6
            if np.isfinite(sse):
                return sse
            else:
                return np.inf
        except Exception as e:
            if verbose:
                print("Residual computation failed:", e)
            return np.inf#np.full(observed.shape[0] * observed.shape[1], 1e6)
    
    return sse_fun

data=pd.read_csv('../Lotka-Volterra/noise_data/1.0_0.csv')

print(data)

x = data[['x','y']]
t = pd.Series(data[['t']].t)

dt = t.to_numpy()[1] - t.to_numpy()[0]
window_length = 41    # must be odd and <= len(t)
polyorder = 3         # generally 2..5

x_hat = savgol_filter(x['x'], window_length, polyorder, delta=dt, mode='interp')
y_hat = savgol_filter(x['y'], window_length, polyorder, delta=dt, mode='interp')

#x['x'] = x_hat
#x['y'] = y_hat

x_dot = savgol_filter(x['x'], window_length, polyorder, deriv=1, delta=dt, mode='interp')
y_dot = savgol_filter(x['y'], window_length, polyorder, deriv=1, delta=dt, mode='interp')

dx = [pd.DataFrame(data = {'x': x_hat, 'y': y_hat}),pd.DataFrame(data = {'x': x_dot, 'y': y_dot})]

print(x)

print(t)

print(dx)

     Unnamed: 0          x         y     t        dx        dy
0             0  10.759572  4.854358   0.0 -0.057780 -0.934931
1             1  10.545703  4.355556   0.5 -0.083852 -0.799113
2             2  10.296010  1.267922   1.0 -0.096984 -0.671264
3             3  11.017958  5.510302   1.5 -0.087758 -0.555217
4             4  10.403667  2.789376   2.0 -0.052200 -0.454869
..          ...        ...       ...   ...       ...       ...
155         155  18.015525  1.933623  77.5  1.706613 -0.205971
156         156  16.428998  3.309608  78.0  1.987492 -0.472696
157         157  17.467785  0.025648  78.5  2.279009 -0.785711
158         158  21.166165  0.517199  79.0  2.579982 -1.140524
159         159  21.056911  0.092972  79.5  2.889226 -1.532002

[160 rows x 6 columns]
             x         y
0    10.759572  4.854358
1    10.545703  4.355556
2    10.296010  1.267922
3    11.017958  5.510302
4    10.403667  2.789376
..         ...       ...
155  18.015525  1.933623
156  16.428998  3.30

## Regular workflow: Lambdify + Pythonic Integration RK2

In [3]:
all_symbols = {}
expr_list = []
atomd_list = []
fit_key_list = []
# Collect all symbols across all equations
n_eq = 0
trees = ['a*x + b*x*y', 'c*y + d*x*y']
def extract_symbols(tree):
    """Extract variables and parameters from a tree."""
    expr = sympify(str(tree))
    atoms = expr.atoms()
    atomd = {a.name: a for a in atoms if a.is_Symbol}
    return expr, atomd
for tree in trees:
    expr, atomd = extract_symbols(tree)
    expr_list.append(expr)
    atomd_list.append(atomd)
    all_symbols.update(atomd)
    fit_key_list.append(str(tree))
    n_eq +=1
fit_key = tuple(fit_key_list)
# Sort symbols: variables first, then parameters
# Using self.variables/self.parameters as order reference
variables = [Symbol(v) for v in ['x','y']]# Always use all variables if v in all_symbols]
parameters = [all_symbols[p] for p in ['a','b','c','d'] if p in list(all_symbols.keys())]

all_sorted_symbols = variables + parameters
#all_sorted_atoms = [atomd_list[0][s] if s in atomd_list[0] else sympify(s) for s in all_sorted_symbols]
# Lambdify all equations using the **same inputs**
lambdas = []
for expr in expr_list:
    try:
        f_lam = lambdify(
            (variables , parameters),
            expr,
            ["numpy", {"fac": scipy.special.factorial}]
            #modules='jax'
        )
        lambdas.append(f_lam)
    except Exception as e:
        print("Lambdify error:", e)
        lambdas.append(lambda *args: np.nan)
def ODE(v,p):
    return np.array([f(v,p) for f in lambdas])

ds = 'd0'

this_x = x.iloc[:, :n_eq]
xmat = this_x.to_numpy(dtype=float64)

this_xhat = dx[0].iloc[:, :n_eq]
xhatmat = this_xhat.to_numpy(dtype=float64)

this_dx = dx[1].iloc[:, :n_eq]
dxmat = this_dx.to_numpy(dtype=float64)

this_t = t.to_numpy(dtype=float64)
cols = this_x.columns[:n_eq]  # take first n_equations columns
x0 = this_x.loc[0, cols].to_numpy(dtype=float64)
p0 = [1. for p in parameters]

def rk2( f, t_eval, y0,params):
    y = np.zeros((len(t_eval), len(y0)),dtype=float64)
    y[0] = y0
    for i in range(len(t_eval) - 1):
        h = t_eval[i+1] - t_eval[i]
        
        k1 = f(y[i],params)
        k2 = f(y[i] + h * k1 / 2,params)
        y[i+1] = y[i] + 0.5 * h * (k1 + k2)
        
        #y[i+1] = y[i] + h * f(y[i],params)
    return y

sse_fun = make_sse_function_from_ode(ODE, this_t, rk2, x0_fixed=x0, param_fixed=p0,
                                                    optimize_x0=True, optimize_params=True, observed=xmat)

In [4]:
%%time
for i in range(100):
    def sse_deriv(params, ODE_func, X, dX_dt):
        model_dx = np.array([ODE_func(X[i], params) for i in range(len(X))])
        return np.sum((dX_dt - model_dx)**2.)
    df_fit_par = minimize(
            sse_deriv,
            x0=np.asarray(p0, dtype=float64),
            args=(ODE, xhatmat, dxmat),
            method='Powell'  # or 'Nelder-Mead' if you want derivative-free
        ).x
    res = minimize(sse_fun, x0=df_fit_par, method='Powell')

print(df_fit_par)
print(res)

[ 0.09845035 -0.01945074 -0.36362961  0.0178495 ]
 message: Maximum number of function evaluations has been exceeded.
 success: False
  status: 1
     fun: inf
       x: [ 9.845e-02 -1.945e-02 -3.636e-01  1.785e-02]
     nit: 307
   direc: [[ 1.000e+00  0.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  1.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  1.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  0.000e+00  1.000e+00]]
    nfev: 4000
CPU times: user 18.8 s, sys: 1.69 ms, total: 18.8 s
Wall time: 18.8 s


## Lambdify + Pythonic RK4

In [5]:
def rk4(f, t_eval, y0, params):
    n_steps = len(t_eval)
    n_vars = len(y0)
    y = np.zeros((n_steps, n_vars), dtype=float64)
    y[0] = y0
    
    for i in range(n_steps - 1):
        h = t_eval[i+1] - t_eval[i]
        k1 = f(y[i], params)
        k2 = f(y[i] + 0.5*h*k1, params)
        k3 = f(y[i] + 0.5*h*k2, params)
        k4 = f(y[i] + h*k3, params)
        y[i+1] = y[i] + (h/6.0)*(k1 + 2*k2 + 2*k3 + k4)
    
    return y

sse_fun = make_sse_function_from_ode(ODE, this_t, rk4, x0_fixed=x0, param_fixed=p0,
                                                    optimize_x0=True, optimize_params=True, observed=xmat)

In [6]:
%%time
x0=[]
for i in range(100):
    def sse_deriv(params, ODE_func, X, dX_dt):
        model_dx = np.array([ODE_func(X[i], params) for i in range(len(X))])
        return np.sum((dX_dt - model_dx)**2.)
    df_fit_par = minimize(
            sse_deriv,
            x0=np.asarray(p0, dtype=float64),
            args=(ODE, xhatmat, dxmat),
            method='Powell'  # or 'Nelder-Mead' if you want derivative-free
        ).x
    res = minimize(sse_fun, x0=df_fit_par, method='Powell')

print(df_fit_par)
print(res)

[ 0.09845035 -0.01945074 -0.36362961  0.0178495 ]
 message: Maximum number of function evaluations has been exceeded.
 success: False
  status: 1
     fun: inf
       x: [ 9.845e-02 -1.945e-02 -3.636e-01  1.785e-02]
     nit: 307
   direc: [[ 1.000e+00  0.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  1.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  1.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  0.000e+00  1.000e+00]]
    nfev: 4000
CPU times: user 18.7 s, sys: 2.91 ms, total: 18.7 s
Wall time: 18.7 s


## Precompiled Lambda + JIT RK2

In [7]:
def make_numba_ODE(exprs, variables, parameters):
    n_eq = len(exprs)
    var_names = [str(v) for v in variables]
    par_names = [str(p) for p in parameters]
    
    code = "def ODE(v, p):\n"
    # map v -> variables
    for i, var in enumerate(var_names):
        code += f"    {var} = v[{i}]\n"
    # map p -> parameters
    for i, par in enumerate(par_names):
        code += f"    {par} = p[{i}]\n"
    # allocate output
    code += f"    out = np.empty({n_eq}, dtype=np.float64)\n"
    # assign expressions
    for i, expr in enumerate(exprs):
        code += f"    out[{i}] = {ccode(expr)}\n"
    code += "    return out\n"
    ns = {"np": np}
    exec(code, ns)
    return njit(cache=False)(ns["ODE"])
    
ODE = make_numba_ODE(expr_list, variables, parameters)

@njit(cache=False)
def RK2( f, t_eval, y0,params):
    y = np.zeros((len(t_eval), len(y0)),dtype=float64)
    y[0] = y0
    for i in range(len(t_eval) - 1):
        h = t_eval[i+1] - t_eval[i]
        
        k1 = f(y[i],params)
        k2 = f(y[i] + h * k1 / 2,params)
        y[i+1] = y[i] + 0.5 * h * (k1 + k2)
        
        #y[i+1] = y[i] + h * f(y[i],params)
    return y

sse_fun = make_sse_function_from_ode(ODE, this_t, RK2, x0_fixed=x0, param_fixed=p0,
                                                    optimize_x0=True, optimize_params=True, observed=xmat)

In [8]:
%%time
x0=[]
for i in range(100):
    def sse_deriv(params, ODE_func, X, dX_dt):
        model_dx = np.array([ODE_func(X[i], params) for i in range(len(X))])
        return np.sum((dX_dt - model_dx)**2.)
    df_fit_par = minimize(
            sse_deriv,
            x0=np.asarray(p0, dtype=float64),
            args=(ODE, xhatmat, dxmat),
            method='Powell'  # or 'Nelder-Mead' if you want derivative-free
        ).x
    res = minimize(sse_fun, x0=df_fit_par, method='Powell')

print(df_fit_par)
print(res)

[ 0.09845035 -0.01945074 -0.36362961  0.0178495 ]
 message: Maximum number of function evaluations has been exceeded.
 success: False
  status: 1
     fun: inf
       x: [ 9.845e-02 -1.945e-02 -3.636e-01  1.785e-02]
     nit: 307
   direc: [[ 1.000e+00  0.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  1.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  1.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  0.000e+00  1.000e+00]]
    nfev: 4000
CPU times: user 12.2 s, sys: 140 ms, total: 12.3 s
Wall time: 12.3 s


In [9]:
@njit(cache=False)
def RK4(f, t_eval, y0, params):
    n_steps = len(t_eval)
    n_vars = len(y0)
    y = np.zeros((n_steps, n_vars), dtype=float64)
    y[0] = y0

    for i in range(n_steps - 1):
        h = t_eval[i+1] - t_eval[i]
        k1 = f(y[i], params)
        k2 = f(y[i] + 0.5*h*k1, params)
        k3 = f(y[i] + 0.5*h*k2, params)
        k4 = f(y[i] + h*k3, params)
        y[i+1] = y[i] + (h/6.0)*(k1 + 2*k2 + 2*k3 + k4)

    return y

sse_fun = make_sse_function_from_ode(ODE, this_t, RK4, x0_fixed=x0, param_fixed=p0,
                                                    optimize_x0=True, optimize_params=True, observed=xmat)

In [10]:
%%time
x0=[]
for i in range(100):
    def sse_deriv(params, ODE_func, X, dX_dt):
        model_dx = np.array([ODE_func(X[i], params) for i in range(len(X))])
        return np.sum((dX_dt - model_dx)**2.)
    df_fit_par = minimize(
            sse_deriv,
            x0=np.asarray(p0, dtype=float64),
            args=(ODE, xhatmat, dxmat),
            method='Powell'  # or 'Nelder-Mead' if you want derivative-free
        ).x
    res = minimize(sse_fun, x0=df_fit_par, method='Powell')

print(df_fit_par)
print(res)

[ 0.09845035 -0.01945074 -0.36362961  0.0178495 ]
 message: Maximum number of function evaluations has been exceeded.
 success: False
  status: 1
     fun: inf
       x: [ 9.845e-02 -1.945e-02 -3.636e-01  1.785e-02]
     nit: 307
   direc: [[ 1.000e+00  0.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  1.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  1.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  0.000e+00  1.000e+00]]
    nfev: 4000
CPU times: user 12 s, sys: 122 ms, total: 12.1 s
Wall time: 12.1 s
